In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/content/T2ICount-Implementation")
REPO_URL = "https://github.com/bqa100507-spec/T2ICount-Implementation.git"
BRANCH = "rich-prompt"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=REPO_DIR,
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"Path exists but is not a Git repository: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

Working directory: /content/T2ICount-Implementation


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

In [3]:
import os
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

ASSET_ROOT = "/content/T2ICount-assets"
DRIVE_ASSET_ROOT = "/content/drive/MyDrive/T2ICount-assets"

os.environ["T2ICOUNT_ASSET_ROOT"] = ASSET_ROOT
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Reuse the baseline notebook's Drive ZIP -> local SSD asset bootstrap.
DRIVE_ZIP = Path("/content/drive/MyDrive/T2ICount-assets.zip")
LOCAL_ZIP = Path("/content/T2ICount-assets.zip")
ASSET_ROOT_PATH = Path(ASSET_ROOT)
DRIVE_DIAGNOSTIC_ROOT = (
    Path(DRIVE_ASSET_ROOT) / "checkpoints/rich_prompt/diagnostic"
)
DRIVE_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRED_ASSETS = {
    "CLIP": (
        ASSET_ROOT_PATH / "pretrained/clip-vit-large-patch14",
        "directory",
    ),
    "Stable Diffusion": (
        ASSET_ROOT_PATH / "pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt",
        "file",
    ),
    "FSC147": (ASSET_ROOT_PATH / "datasets/FSC147", "directory"),
    "T2ICount checkpoint": (
        ASSET_ROOT_PATH / "checkpoints/official/best_model_paper.pth",
        "file",
    ),
}


def check_assets(verbose=True):
    status = {}
    for name, (path, kind) in REQUIRED_ASSETS.items():
        status[name] = path.is_dir() if kind == "directory" else path.is_file()

    if verbose:
        print("\nAsset status:")
        for name, ok in status.items():
            print(f"  [{'OK' if ok else 'MISSING'}] {name}")
            if not ok:
                print(f"       {REQUIRED_ASSETS[name][0]}")

    return all(status.values())


def human_size(num_bytes):
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} PB"


def copy_with_progress(src, dst, chunk_size=64 * 1024 * 1024):
    total = src.stat().st_size
    dst.parent.mkdir(parents=True, exist_ok=True)
    with src.open("rb") as fsrc, dst.open("wb") as fdst:
        with tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Copy ZIP Drive -> local",
        ) as progress:
            while True:
                chunk = fsrc.read(chunk_size)
                if not chunk:
                    break
                fdst.write(chunk)
                progress.update(len(chunk))


if check_assets():
    print("\nLocal assets already complete:", ASSET_ROOT_PATH)
else:
    if not DRIVE_ZIP.is_file():
        raise FileNotFoundError(
            "Cannot find /content/drive/MyDrive/T2ICount-assets.zip. "
            "Upload the baseline asset ZIP to My Drive first."
        )

    drive_zip_size = DRIVE_ZIP.stat().st_size
    print("\nDrive ZIP found.")
    print("ZIP size:", human_size(drive_zip_size))

    local_zip_valid = (
        LOCAL_ZIP.is_file() and LOCAL_ZIP.stat().st_size == drive_zip_size
    )

    if local_zip_valid:
        print("\nLocal ZIP already exists with matching size.")
        print("Skipping Drive copy:", LOCAL_ZIP)
    else:
        # Remove only an incomplete local ZIP, never anything on Drive.
        if LOCAL_ZIP.exists():
            print("\nRemoving incomplete local ZIP:", LOCAL_ZIP)
            LOCAL_ZIP.unlink()

        free_space = shutil.disk_usage("/content").free
        print("Free local disk:", human_size(free_space))
        if free_space < drive_zip_size + 5 * 1024**3:
            raise RuntimeError("Not enough local disk space to safely copy the asset archive.")

        print("\nCopying ZIP from Google Drive to Colab SSD...")
        start = time.perf_counter()
        copy_with_progress(DRIVE_ZIP, LOCAL_ZIP)
        elapsed = time.perf_counter() - start
        print(f"\nZIP copied in {elapsed / 60:.1f} minutes.")

    print("\nChecking ZIP integrity...")
    if not zipfile.is_zipfile(LOCAL_ZIP):
        raise RuntimeError(f"Local archive is not a valid ZIP: {LOCAL_ZIP}")

    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = zf.infolist()
        names = [info.filename.replace('\\', '/') for info in members]
        has_root_folder = any(
            name.startswith("T2ICount-assets/") for name in names
        )
        total_uncompressed = sum(info.file_size for info in members)

    print("Uncompressed size:", human_size(total_uncompressed))
    print("Files in archive:", len(members))

    if has_root_folder:
        extract_to = Path("/content")
    else:
        ASSET_ROOT_PATH.mkdir(parents=True, exist_ok=True)
        extract_to = ASSET_ROOT_PATH

    print(
        "\nArchive layout:",
        "contains T2ICount-assets/"
        if has_root_folder
        else "contents start inside asset root",
    )
    print("Extract target:", extract_to)
    print("\nExtracting on Colab local SSD...")

    start = time.perf_counter()
    result = subprocess.run(
        ["unzip", "-q", "-o", str(LOCAL_ZIP), "-d", str(extract_to)],
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"unzip failed with return code {result.returncode}")

    elapsed = time.perf_counter() - start
    print(f"Extraction finished in {elapsed / 60:.1f} minutes.")
    print("\nValidating extracted assets...")
    if not check_assets():
        raise RuntimeError(
            "Extraction finished but required assets are still missing."
        )

    if LOCAL_ZIP.exists():
        print("\nRemoving local ZIP to recover disk space...")
        LOCAL_ZIP.unlink()

print("\n========================================")
print("LOCAL ASSET BOOTSTRAP PASSED")
print("========================================")
print("Runtime assets :", ASSET_ROOT_PATH)
print("Drive saves    :", DRIVE_DIAGNOSTIC_ROOT)
print("T2ICOUNT_ASSET_ROOT =", os.environ["T2ICOUNT_ASSET_ROOT"])


Asset status:
  [MISSING] CLIP
       /content/T2ICount-assets/pretrained/clip-vit-large-patch14
  [MISSING] Stable Diffusion
       /content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
  [MISSING] FSC147
       /content/T2ICount-assets/datasets/FSC147
  [MISSING] T2ICount checkpoint
       /content/T2ICount-assets/checkpoints/official/best_model_paper.pth

Drive ZIP found.
ZIP size: 12.96 GB

Removing incomplete local ZIP: /content/T2ICount-assets.zip
Free local disk: 185.69 GB

Copying ZIP from Google Drive to Colab SSD...


Copy ZIP Drive -> local:   0%|          | 0.00/13.0G [00:00<?, ?B/s]


ZIP copied in 3.4 minutes.

Checking ZIP integrity...
Uncompressed size: 17.86 GB
Files in archive: 15781

Archive layout: contains T2ICount-assets/
Extract target: /content

Extracting on Colab local SSD...
Extraction finished in 2.9 minutes.

Validating extracted assets...

Asset status:
  [OK] CLIP
  [OK] Stable Diffusion
  [OK] FSC147
  [OK] T2ICount checkpoint

Removing local ZIP to recover disk space...

LOCAL ASSET BOOTSTRAP PASSED
Runtime assets : /content/T2ICount-assets
Drive saves    : /content/drive/MyDrive/T2ICount-assets/checkpoints/rich_prompt/diagnostic
T2ICOUNT_ASSET_ROOT = /content/T2ICount-assets


In [4]:
%cd /content/T2ICount-Implementation

!git checkout rich-prompt
!git pull origin rich-prompt
!pip install -q -r requirements-colab.txt

/content
Already on 'rich-prompt'
Your branch is up to date with 'origin/rich-prompt'.
From https://github.com/bqa100507-spec/T2ICount-Implementation
 * branch            rich-prompt -> FETCH_HEAD
Already up to date.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 136.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 43.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 79.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import torch
import pytorch_lightning
import omegaconf
import transformers
import kornia

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("pytorch_lightning:", pytorch_lightning.__version__)

torch: 2.11.0+cu128
cuda: True
pytorch_lightning: 2.6.5


In [6]:
from pathlib import Path

PROMPT_BANK = Path(
    "/content/drive/MyDrive/T2ICount-assets/prompts/"
    "fsc147_train1000_seed3407_gemma4_v3.json"
)
if not PROMPT_BANK.is_file():
    raise FileNotFoundError(
        f"Upload/copy fsc147_train1000_seed3407_gemma4_v3.json to {PROMPT_BANK.parent}"
    )

# Expected:
# selected_sample_count=1000
# selected_image_fingerprint=sha256:dd96b36bf15013e194b1a8ece06452a19822aae028fc00c0de019cbb7a311f24
!python tools/validate_rich_prompt_training_bank.py \
  --rich-prompt-bank /content/drive/MyDrive/T2ICount-assets/prompts/fsc147_train1000_seed3407_gemma4_v3.json \
  --data-dir /content/T2ICount-assets/datasets/FSC147 \
  --train-samples 1000 \
  --train-subset-seed 3407

Rich prompt bank validation passed.
selected_sample_count=1000
selected_image_fingerprint=sha256:dd96b36bf15013e194b1a8ece06452a19822aae028fc00c0de019cbb7a311f24
prompt_bank_fingerprint=sha256:c3ec587b37fd9f351cdedbce888b26e0ba7a020de956c4ef18202a1268e7e32f
protocol_version=rich-prompt-phase1-v3


In [8]:
%cd /content/T2ICount-Implementation

!python tools/evaluate_rich_prompts.py \
  --asset-root /content/T2ICount-assets \
  --metadata FSC-147-S.json \
  --prompt-bank /content/drive/MyDrive/T2ICount-assets/prompts/fsc147s_gemma4_26b_v3.json \
  --checkpoint /content/drive/MyDrive/T2ICount-assets/checkpoints/rich_prompt/pilot_1000x10_lambda0p1_rerun/rich_prompt_1000x10_lambda0p1_rerun/best_model_9.pth \
  --output-dir /content/drive/MyDrive/T2ICount-assets/outputs/rich_prompt_phase2c_lambda0p1 \
  --batch-size 16 \
  --device cuda \
  --overwrite

/content
checkpoint path: /content/drive/MyDrive/T2ICount-assets/checkpoints/rich_prompt/pilot_1000x10_lambda0p1_rerun/rich_prompt_1000x10_lambda0p1_rerun/best_model_9.pth
prompt bank path: /content/drive/MyDrive/T2ICount-assets/prompts/fsc147s_gemma4_26b_v3.json
protocol version: rich-prompt-phase1-v3
generator: gemma-4-26b-a4b-it
sample count: 230
device: cuda
patch batch size: 16
Loading model from /content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
LatentDiffusion: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
2026-09-07 11:23:56.765381: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF